# 05 - Instrumental variables

Instrumental variables can help when treatment is confounded by unobserved factors, provided a valid instrument exists.


## Causal question
State the causal question this notebook answers before reading numeric output.

## Causal setup
- Treatment: define the treatment variable and intervention of interest.
- Outcome: define the outcome variable being affected by treatment.
- Covariates: list observed confounders included in the design/diagnostics.
- Unit of analysis: specify the observational unit used in this notebook.

## Estimand
Specify the target estimand (ATE, ATT, CATE, etc.) and how it maps to model coefficients.

## Identification assumptions
Enumerate the identification assumptions required for a causal interpretation (for example ignorability, overlap, no interference).

## Uncertainty and limitations
Report interval estimates, sensitivity checks, and at least one limitation of the design.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Step execution
This cell performs the next computation. Read the result and tie it back to the causal setup before moving on.


In [ ]:
import statsmodels.api as sm

from causal_inference_lab.data_generators import make_iv_data
from causal_inference_lab.estimators import difference_in_means

dataset = make_iv_data(n=5_000, seed=99)
data = dataset.data

data.head()


## Naive estimate

Treatment is affected by an unobserved confounder, so the naive estimate is biased.


In [ ]:
naive = difference_in_means(data)
print(f"Naive estimate: {naive.estimate:.3f}")
print(f"True effect:    {dataset.true_ate:.3f}")


## Manual two-stage least squares

Stage 1 predicts treatment using the instrument and observed covariates. Stage 2 regresses the outcome on predicted treatment and covariates.


In [ ]:
first_stage_x = sm.add_constant(data[["instrument", "x"]])
first_stage = sm.OLS(data["treatment"], first_stage_x).fit()
data = data.assign(treatment_hat=first_stage.predict(first_stage_x))

second_stage_x = sm.add_constant(data[["treatment_hat", "x"]])
second_stage = sm.OLS(data["outcome"], second_stage_x).fit()

print(f"First-stage instrument coefficient: {first_stage.params['instrument']:.3f}")
print(f"IV estimate:                         {second_stage.params['treatment_hat']:.3f}")
print(f"True effect:                         {dataset.true_ate:.3f}")


**Interpretation.** IV can address hidden confounding only if the instrument is valid. The instrument must affect treatment, have no direct path to the outcome, and not be related to the unobserved outcome determinants except through treatment.
